# Model Comparison - 8 Models on Bitcoin Data

**Purpose**: Train and compare all available models to identify best performer

**Models to Compare**:
1. **Neural**: N-BEATS, LSTM, TCN, TFT (Temporal Fusion Transformer)
2. **Classical**: ARIMA, Prophet
3. **ML**: XGBoost, XGBoost

**Methodology** (Correct - No Data Leakage!):
- Split data FIRST (70/15/15 temporal split)
- Generate features AFTER split using StatefulFeatureEngineer
- Same hyperparameters across models
- Financial metrics with transaction costs
- Suspicious result detection

**Expected Runtime**: ~15-30 minutes (depending on epochs)

**Prerequisites**:
- PostgreSQL with 30+ days of data
- GPU recommended (CPU will work but slower)

In [1]:
"""Import libraries and configure training settings."""
from __future__ import annotations

import asyncio
import time
from datetime import datetime, timedelta
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from loguru import logger
from plotly.subplots import make_subplots
from rich.console import Console
from rich.progress import Progress, SpinnerColumn, TextColumn, BarColumn, TimeElapsedColumn
from rich.table import Table

from price_stradamus.config.constants import (
    FEATURE_WARMUP_PERIOD,
    REALISTIC_DIR_ACC_MAX,
    SUSPICIOUS_DIR_ACC,
    SUSPICIOUS_SHARPE,
)
from price_stradamus.config.settings import settings
from price_stradamus.data.database import DatabaseManager
from price_stradamus.data.preprocessor import DataPreprocessor
# Use StatefulFeatureEngineer - NOT the deprecated FeatureEngineer!
from price_stradamus.data.stateful_features import StatefulFeatureEngineer
from price_stradamus.evaluation.metrics import MetricsCalculator
from price_stradamus.evaluation.financial_metrics import TradingCosts
from price_stradamus.models.registry import ModelRegistry
from price_stradamus.utils.feature_selection import CORE_FEATURES  # ← NEW: Import core features
from price_stradamus.visualization.charts import PriceChartVisualizer

# Initialize console for Rich output
console = Console()

# Training configuration
TRAINING_CONFIG = {
    "symbol": "BTCUSDT",
    "timeframe": "1m",
    "days": 120,  # Fetch 4 months of data
    "input_chunk_length": 60,
    "output_chunk_length": 5,
    "n_epochs": 200,
    # Custom splits: 3 months train, 3 hours test, rest validation
    "train_window_days": 90,  # 3 months
    "test_window_hours": 3,   # 3 hours
}

# Trading costs for realistic evaluation
COSTS = TradingCosts(commission_pct=0.001, slippage_pct=0.0005)

# Models to compare
MODELS_TO_COMPARE = [
    "xgboost",
    "xgboost",
]

console.print("[bold cyan]Model Comparison Setup Complete[/bold cyan]")
console.print(f"Models to train: {len(MODELS_TO_COMPARE)}")
console.print(f"Transaction costs: {COSTS.round_trip_cost*100:.2f}% per round trip")
console.print(f"Using CORE features: {len(CORE_FEATURES)} features (no leakage, no redundancy)")
console.print(f"Configuration: {TRAINING_CONFIG}")

2025-12-13 22:27:36.965 | INFO     | price_stradamus.models.registry:decorator:61 - Registered model 'xgboost' -> XGBoostModel
2025-12-13 22:27:36.967 | INFO     | price_stradamus.models.registry:decorator:61 - Registered model 'xgboost' -> XGBoostModel


Model Comparison Setup Complete

Models to train: 2

Transaction costs: 0.30% per round trip

Using CORE features: 19 features (no leakage, no redundancy)

Configuration: {'symbol': 'BTCUSDT', 'timeframe': '1m', 'days': 120, 'input_chunk_length': 60, 
'output_chunk_length': 5, 'n_epochs': 200, 'train_window_days': 90, 'test_window_hours': 3}

In [2]:
"""Load and preprocess Bitcoin data from database.

IMPORTANT: DO NOT generate features here!
Features must be generated AFTER the train/test split.
"""

console.print("[cyan]Loading Bitcoin data...[/cyan]")

db = DatabaseManager(settings.database_url)
await db.initialize()

end_date = datetime.now()
start_date = end_date - timedelta(days=TRAINING_CONFIG["days"])

df = await db.get_ohlcv(
    symbol=TRAINING_CONFIG["symbol"],
    timeframe=TRAINING_CONFIG["timeframe"],
    start_date=start_date,
    end_date=end_date,
)

await db.close()

# Preprocess (validation, missing values only - NO feature generation!)
preprocessor = DataPreprocessor()
df_clean = preprocessor.validate_ohlcv(df)
df_clean = preprocessor.handle_missing_values(df_clean)

# Ensure timestamp is datetime
df_clean["timestamp"] = pd.to_datetime(df_clean["timestamp"], utc=True)

console.print(f"[green]✓ Loaded {len(df_clean)} candles[/green]")
console.print(f"[yellow]⚠️  Features NOT generated yet - will generate AFTER split[/yellow]")

df_clean.tail()

Loading Bitcoin data...

2025-12-13 22:27:37.025 | INFO     | price_stradamus.data.database:__init__:57 - DatabaseManager initialized
2025-12-13 22:27:37.280 | INFO     | price_stradamus.data.database:initialize:82 - Database engine and session maker initialized (pool_size=10, max_overflow=20)
2025-12-13 22:27:37.281 | DEBUG    | price_stradamus.data.database:get_ohlcv:196 - Querying OHLCV data for BTCUSDT 1m
2025-12-13 22:27:40.790 | INFO     | price_stradamus.data.database:get_ohlcv:251 - Retrieved 164058 OHLCV records for BTCUSDT 1m
2025-12-13 22:27:40.848 | INFO     | price_stradamus.data.database:close:92 - Database connections closed
2025-12-13 22:27:40.850 | INFO     | price_stradamus.data.preprocessor:__init__:60 - DataPreprocessor initialized (normalization=minmax)
2025-12-13 22:27:40.891 | DEBUG    | price_stradamus.data.preprocessor:validate_ohlcv:138 - OHLCV validation passed for 164058 rows
2025-12-13 22:27:40.895 | DEBUG    | price_stradamus.data.preprocessor:handle_missing_values:167 - No missin

✓ Loaded 164058 candles

⚠️  Features NOT generated yet - will generate AFTER split

,timestamp,open,high,low,close,volume,quote_volume,num_trades
164053,2025-12-07 13:41:00+00:00,88958.80,88984.00,88958.80,88968.64,5.90303,525252.524440,1501
164054,2025-12-07 13:42:00+00:00,88968.64,88996.26,88968.63,88968.67,7.48373,665946.211646,1806
164055,2025-12-07 13:43:00+00:00,88968.67,88997.38,88948.00,88948.00,5.34612,475661.877739,2103
164056,2025-12-07 13:44:00+00:00,88948.00,88977.49,88948.00,88965.36,5.84777,520277.777900,1724
164057,2025-12-07 13:45:00+00:00,88965.37,88999.97,88965.36,88983.01,1.67235,148805.795654,1316


In [3]:
"""Create temporal train/validation/test splits and generate features.

CORRECT ORDER:
1. Split RAW OHLCV data first
2. Create StatefulFeatureEngineer
3. fit_transform on training data
4. transform on validation/test data
5. Filter to CORE_FEATURES only (19 features - no leakage, no redundancy)
"""

# Step 1: Split RAW data FIRST (before features!)
# Custom splits: 3 months train, 3 hours test, rest validation
test_window_minutes = TRAINING_CONFIG["test_window_hours"] * 60  # 3 hours = 180 minutes
train_window_minutes = TRAINING_CONFIG["train_window_days"] * 24 * 60  # 90 days = 129,600 minutes

# Calculate splits from the END of data (most recent)
test_size = test_window_minutes  # Last 180 rows
train_size = min(train_window_minutes, len(df_clean) - test_size - FEATURE_WARMUP_PERIOD)  # 3 months or available
val_size = len(df_clean) - train_size - test_size  # Remainder

# Split data (temporal order: train -> val -> test)
train_df = df_clean.iloc[:train_size].copy()
val_df = df_clean.iloc[train_size:train_size + val_size].copy()
test_df = df_clean.iloc[train_size + val_size:].copy()

console.print("\n[bold]Step 1: Split RAW data (before features)[/bold]")
console.print(f"Train window: {train_size} rows ({train_size/1440:.1f} days)")
console.print(f"Val window: {val_size} rows ({val_size/1440:.1f} days)")
console.print(f"Test window: {test_size} rows ({test_size/60:.1f} hours)")

# Step 2: Generate features using StatefulFeatureEngineer
console.print("\n[bold]Step 2: Generate features (fit on train, transform val/test)[/bold]")

engineer = StatefulFeatureEngineer(warmup_period=FEATURE_WARMUP_PERIOD)

# fit_transform on training data - learns statistics
train_features = engineer.fit_transform(train_df)

# transform on val/test - uses training statistics (NO leakage!)
val_features = engineer.transform(val_df, drop_warmup=False)
test_features = engineer.transform(test_df, drop_warmup=False)

# Step 3: Filter to CORE_FEATURES only (19 features)
console.print("\n[bold]Step 3: Filter to CORE features (no leakage, no redundancy)[/bold]")

# Keep OHLCV columns + core features
keep_cols = ["open", "high", "low", "close", "volume", "timestamp"] + [
    f for f in CORE_FEATURES if f in train_features.columns
]

train_features = train_features[keep_cols].copy()
val_features = val_features[keep_cols].copy()
test_features = test_features[keep_cols].copy()

# Check for missing core features
missing_core = [f for f in CORE_FEATURES if f not in train_features.columns]
if missing_core:
    console.print(f"[yellow]⚠️  Missing core features: {missing_core}[/yellow]")

# Convert to Darts TimeSeries
train_ts = engineer.to_darts_timeseries(train_features, value_cols=["close"])
val_ts = engineer.to_darts_timeseries(val_features, value_cols=["close"])
test_ts = engineer.to_darts_timeseries(test_features, value_cols=["close"])

# Display split info
table = Table(title="Data Splits (Temporal, CORE Features, No Leakage)")
table.add_column("Set", style="cyan")
table.add_column("Raw Size", style="yellow")
table.add_column("With Features", style="green")
table.add_column("Start", style="dim")
table.add_column("End", style="dim")

table.add_row(
    "Train",
    str(len(train_df)),
    str(len(train_ts)),
    str(train_ts.time_index[0])[:16],
    str(train_ts.time_index[-1])[:16],
)
table.add_row(
    "Validation",
    str(len(val_df)),
    str(len(val_ts)),
    str(val_ts.time_index[0])[:16],
    str(val_ts.time_index[-1])[:16],
)
table.add_row(
    "Test",
    str(len(test_df)),
    str(len(test_ts)),
    str(test_ts.time_index[0])[:16],
    str(test_ts.time_index[-1])[:16],
)

console.print(table)

# Display core features
console.print(f"\n[green]✓ Using {len([f for f in CORE_FEATURES if f in train_features.columns])} CORE features:[/green]")
console.print(CORE_FEATURES)
console.print(f"\n[green]✓ Feature statistics fit on training data only - NO LEAKAGE![/green]")
console.print(f"[green]✓ No redundant features - correlation <0.95[/green]")

Step 1: Split RAW data (before features)

Train window: 129600 rows (90.0 days)

Val window: 34278 rows (23.8 days)

Test window: 180 rows (3.0 hours)

Step 2: Generate features (fit on train, transform val/test)

...

               Data Splits (Temporal, CORE Features, No Leakage)               
┏━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┓
┃ Set        ┃ Raw Size ┃ With Features ┃ Start            ┃ End              ┃
┡━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━┩
│ Train      │ 129600   │ 129400        │ 2025-08-15 18:48 │ 2025-11-13 15:27 │
│ Validation │ 34278    │ 34278         │ 2025-11-13 15:28 │ 2025-12-07 10:45 │
│ Test       │ 180      │ 180           │ 2025-12-07 10:46 │ 2025-12-07 13:45 │
└────────────┴──────────┴───────────────┴──────────────────┴──────────────────┘

✓ Using 19 CORE features:

[
    'ema_7',
    'ema_200',
    'rsi_14',
    'macd',
    'macd_signal',
    'macd_hist',
    'roc_10',
    'atr_14',
    'std_20',
    'bb_percent',
    'hist_volatility_20',
    'obv',
    'volume_ratio',
    'volume_price_trend',
    'volume_change_pct',
    'high_low_range',
    'returns',
    'adx_14',
    'supertrend_direction'
]

✓ Feature statistics fit on training data only - NO LEAKAGE!

✓ No redundant features - correlation <0.95

## Model Training

Training all 8 models on the same data with identical configurations.

**Note**: This cell takes 15-30 minutes depending on hardware. Progress bars show status.

In [4]:
"""Train all models and collect results."""

def train_single_model(
    model_name: str,
    train_ts,
    val_ts,
    config: dict[str, Any],
) -> dict[str, Any]:
    """Train a single model and return results.
    
    Returns:
        Dict with model_name, model_instance, training_time, and any errors
    """
    start_time = time.time()
    
    try:
        # Create model
        model = ModelRegistry.create_model(
            model_name,
            input_chunk_length=config["input_chunk_length"],
            output_chunk_length=config["output_chunk_length"],
            n_epochs=config["n_epochs"],
        )

        # Train
        model.fit(train_ts, val_ts)

        training_time = time.time() - start_time

        return {
            "model_name": model_name,
            "model": model,
            "training_time": training_time,
            "status": "success",
            "error": None,
        }
    
    except Exception as e:
        training_time = time.time() - start_time
        logger.error(f"Failed to train {model_name}: {e}")
        return {
            "model_name": model_name,
            "model": None,
            "training_time": training_time,
            "status": "failed",
            "error": str(e),
        }

# Train all models with progress tracking
training_results = []

with Progress(
    SpinnerColumn(),
    TextColumn("[progress.description]{task.description}"),
    BarColumn(),
    TextColumn("[progress.percentage]{task.percentage:>3.0f}%"),
    TimeElapsedColumn(),
    console=console,
) as progress:
    
    task = progress.add_task(
        "[cyan]Training models...",
        total=len(MODELS_TO_COMPARE),
    )
    
    for model_name in MODELS_TO_COMPARE:
        progress.update(task, description=f"[cyan]Training {model_name}...")
        
        result = train_single_model(
            model_name,
            train_ts,
            val_ts,
            TRAINING_CONFIG,
        )
        
        training_results.append(result)
        progress.advance(task)
        
        if result["status"] == "success":
            console.print(f"[green]✓ {model_name} trained in {result['training_time']:.1f}s[/green]")
        else:
            console.print(f"[red]✗ {model_name} failed: {result['error']}[/red]")

# Summary
successful = [r for r in training_results if r["status"] == "success"]
failed = [r for r in training_results if r["status"] == "failed"]

console.print(f"\n[bold]Training Summary:[/bold]")
console.print(f"  Successful: {len(successful)}/{len(MODELS_TO_COMPARE)}")
console.print(f"  Failed: {len(failed)}/{len(MODELS_TO_COMPARE)}")
console.print(f"  Total time: {sum(r['training_time'] for r in training_results):.1f}s")

Output()

2025-12-13 22:27:49.487 | DEBUG    | price_stradamus.models.base:__init__:72 - Initialized XGBoostModel (input=60, output=5)
2025-12-13 22:27:49.489 | INFO     | price_stradamus.models.ml.xgboost:__init__:91 - Initialized XGBoostModel (n_estimators=100, max_depth=6, learning_rate=0.1)
2025-12-13 22:27:49.493 | INFO     | price_stradamus.models.registry:create_model:132 - Created model instance: xgboost


2025-12-13 22:27:49.631 | INFO     | price_stradamus.models.ml.xgboost:fit:121 - Training xgboost on 129400 timesteps (validation: 34278 timesteps)
2025-12-13 22:27:49.632 | INFO     | price_stradamus.models.ml.xgboost:fit:128 - Using tree_method: auto
2025-12-13 22:27:49.634 | INFO     | price_stradamus.models.ml.xgboost:fit:148 - Starting training...


2025-12-13 22:28:00.356 | INFO     | price_stradamus.models.ml.xgboost:fit:156 - Training complete for xgboost


✓ xgboost trained in 10.9s

...

✓ xgboost trained in 11.5s

Training Summary:

Successful: 2/2

Failed: 0/2

Total time: 22.4s

In [5]:
"""Evaluate all successfully trained models on test set with financial metrics."""

evaluation_results = []

for result in training_results:
    if result["status"] != "success":
        continue
    
    model_name = result["model_name"]
    model = result["model"]
    
    try:
        # Predict on test set
        predictions = model.predict(n=len(test_ts), series=train_ts)
        
        # Calculate metrics including financial
        actuals = test_ts.values().flatten()
        preds = predictions.values().flatten()[:len(actuals)]
        
        metrics = MetricsCalculator.calculate_all_with_financial(
            actuals, preds, include_financial=True, trading_costs=COSTS
        )
        
        evaluation_results.append({
            "model_name": model_name,
            "metrics": metrics,
            "predictions": preds,
            "actuals": actuals,
        })
        
    except Exception as e:
        logger.error(f"Failed to evaluate {model_name}: {e}")

# Create results table
table = Table(title=f"Model Performance (Test Set, Costs: {COSTS.round_trip_cost*100:.2f}%)")
table.add_column("Model", style="cyan")
table.add_column("MAE", style="yellow")
table.add_column("RMSE", style="yellow")
table.add_column("Dir Acc %", style="green")
table.add_column("Sharpe", style="magenta")
table.add_column("Net P&L", style="green")

# Sort by RMSE (lower is better)
evaluation_results_sorted = sorted(
    evaluation_results,
    key=lambda x: x["metrics"]["rmse"],
)

for result in evaluation_results_sorted:
    m = result["metrics"]
    table.add_row(
        result["model_name"],
        f"{m['mae']:.2f}",
        f"{m['rmse']:.2f}",
        f"{m['directional_accuracy']:.2f}",
        f"{m['fin_sharpe_ratio']:.2f}",
        f"${m['fin_net_pnl']:.2f}",
    )

console.print(table)

# Highlight best model
best_model = evaluation_results_sorted[0]
console.print(f"\n[bold green]🏆 Best Model: {best_model['model_name']}[/bold green]")
console.print(f"   RMSE: {best_model['metrics']['rmse']:.2f} USDT")
console.print(f"   Directional Accuracy: {best_model['metrics']['directional_accuracy']:.2f}%")
console.print(f"   Sharpe Ratio: {best_model['metrics']['fin_sharpe_ratio']:.2f}")
console.print(f"   Net P&L: ${best_model['metrics']['fin_net_pnl']:.2f}")

# Check for suspicious results
for result in evaluation_results_sorted:
    m = result["metrics"]
    if m['directional_accuracy'] > SUSPICIOUS_DIR_ACC:
        console.print(f"\n[bold red]⚠️  WARNING: {result['model_name']} has suspicious accuracy ({m['directional_accuracy']:.1f}%)[/bold red]")
    if m['fin_sharpe_ratio'] > SUSPICIOUS_SHARPE:
        console.print(f"\n[bold red]⚠️  WARNING: {result['model_name']} has suspicious Sharpe ({m['fin_sharpe_ratio']:.2f})[/bold red]")

2025-12-13 22:28:11.912 | WARNING  | price_stradamus.models.base:validate_predict_params:220 - Requested 180 steps but model is configured for 5 steps
2025-12-13 22:28:11.913 | DEBUG    | price_stradamus.models.ml.xgboost:predict:183 - Predicting 180 steps ahead
2025-12-13 22:28:11.913 | INFO     | price_stradamus.models.ml.xgboost:predict:192 - Using iterative prediction: 180 steps in chunks of 5


2025-12-13 22:28:11.919 | DEBUG    | price_stradamus.models.ml.xgboost:predict:216 - Iteration 1: predicted 5 steps, 175 remaining
2025-12-13 22:28:11.924 | DEBUG    | price_stradamus.models.ml.xgboost:predict:216 - Iteration 2: predicted 5 steps, 170 remaining
2025-12-13 22:28:11.930 | DEBUG    | price_stradamus.models.ml.xgboost:predict:216 - Iteration 3: predicted 5 steps, 165 remaining
2025-12-13 22:28:11.934 | DEBUG    | price_stradamus.models.ml.xgboost:predict:216 - Iteration 4: predicted 5 steps, 160 remaining
2025-12-13 22:28:11.938 | DEBUG    | price_stradamus.models.ml.xgboost:predict:216 - Iteration 5: predicted 5 steps, 155 remaining
2025-12-13 22:28:11.943 | DEBUG    | price_stradamus.models.ml.xgboost:predict:216 - Iteration 6: predicted 5 steps, 150 remaining
2025-12-13 22:28:11.947 | DEBUG    | price_stradamus.models.ml.xgboost:predict:216 - Iteration 7: predicted 5 steps, 145 remaining
2025-12-13 22:28:11.950 | DEBUG    | price_stradamus.models.ml.xgboost:predict:216 

              Model Performance (Test Set, Costs: 0.30%)              
┏━━━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━┓
┃ Model         ┃ MAE      ┃ RMSE     ┃ Dir Acc % ┃ Sharpe ┃ Net P&L ┃
┡━━━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━┩
│ xgboost       │ 12985.53 │ 12986.33 │ 52.51     │ -41.29 │ $-54.70 │
│ xgboost │ 13098.15 │ 13099.00 │ 49.72     │ -41.29 │ $-54.70 │
└───────────────┴──────────┴──────────┴───────────┴────────┴─────────┘

🏆 Best Model: xgboost

RMSE: 12986.33 USDT

Directional Accuracy: 52.51%

Sharpe Ratio: -41.29

Net P&L: $-54.70

⚠️  WARNING: xgboost has suspicious accuracy (52.5%)

⚠️  WARNING: xgboost has suspicious accuracy (49.7%)

In [6]:
"""Create interactive bar charts comparing model metrics."""

# Prepare data for plotting
models = [r["model_name"] for r in evaluation_results_sorted]
mae_values = [r["metrics"]["mae"] for r in evaluation_results_sorted]
rmse_values = [r["metrics"]["rmse"] for r in evaluation_results_sorted]
dir_acc_values = [r["metrics"]["directional_accuracy"] for r in evaluation_results_sorted]

# Create subplot with 3 metrics
fig = make_subplots(
    rows=1,
    cols=3,
    subplot_titles=("MAE (Lower is Better)", "RMSE (Lower is Better)", "Directional Accuracy (Higher is Better)"),
)

# MAE
fig.add_trace(
    go.Bar(
        x=models,
        y=mae_values,
        name="MAE",
        marker_color='cyan',
    ),
    row=1,
    col=1,
)

# RMSE
fig.add_trace(
    go.Bar(
        x=models,
        y=rmse_values,
        name="RMSE",
        marker_color='orange',
    ),
    row=1,
    col=2,
)

# Directional Accuracy
fig.add_trace(
    go.Bar(
        x=models,
        y=dir_acc_values,
        name="Dir Acc",
        marker_color='green',
    ),
    row=1,
    col=3,
)

fig.update_layout(
    title_text="Model Performance Comparison",
    showlegend=False,
    height=500,
    template="plotly_dark",
)

fig.update_xaxes(tickangle=45)
fig.show()

In [7]:
"""Compare actual vs predicted prices for all models using PriceChartVisualizer."""

# Prepare predictions dict (top 5 models only for clarity)
predictions_dict = {
    r["model_name"]: r["predictions"]
    for r in evaluation_results_sorted[:5]
}

# Use PriceChartVisualizer
visualizer = PriceChartVisualizer()
visualizer.plot_prediction_comparison(
    predictions_dict=predictions_dict,
    actual=evaluation_results_sorted[0]["actuals"],
    timestamps=test_ts.time_index,
    title=f"Top 5 Models: Predictions vs Actual ({TRAINING_CONFIG['symbol']})",
    show=True,
)

console.print("[green]✓ Comparison chart displayed[/green]")

✓ Comparison chart displayed

In [8]:
"""Analyze prediction error distributions for each model."""

fig = go.Figure()

for result in evaluation_results_sorted:
    errors = result["predictions"] - result["actuals"]
    
    fig.add_trace(go.Box(
        y=errors,
        name=result["model_name"],
        boxmean='sd',  # Show mean and std dev
    ))

fig.update_layout(
    title="Prediction Error Distribution by Model",
    yaxis_title="Prediction Error (USDT)",
    template="plotly_dark",
    height=600,
    showlegend=True,
)

fig.show()

console.print("[green]✓ Error distribution displayed[/green]")

✓ Error distribution displayed

## Conclusions & Recommendations

Based on the model comparison results above.

In [9]:
"""Generate final recommendations based on comparison."""

console.print("="*70)
console.print(" MODEL COMPARISON - FINAL RECOMMENDATIONS ")
console.print("="*70)

# Overall assessment
console.print(f"\n[bold]METHODOLOGY:[/bold]")
console.print(f"  ✓ Split RAW data first (no feature leakage)")
console.print(f"  ✓ Used StatefulFeatureEngineer (fit on train, transform test)")
console.print(f"  ✓ Transaction costs included ({COSTS.round_trip_cost*100:.2f}% per trade)")

# Best overall
best = evaluation_results_sorted[0]
console.print(f"\n1. BEST OVERALL MODEL: [bold green]{best['model_name'].upper()}[/bold green]")
console.print(f"   - RMSE: {best['metrics']['rmse']:.2f} USDT")
console.print(f"   - MAE: {best['metrics']['mae']:.2f} USDT")
console.print(f"   - Directional Accuracy: {best['metrics']['directional_accuracy']:.2f}%")
console.print(f"   - Sharpe Ratio: {best['metrics']['fin_sharpe_ratio']:.2f}")
console.print(f"   - Net P&L: ${best['metrics']['fin_net_pnl']:.2f}")

# Best for directional accuracy (trading)
best_dir = max(evaluation_results_sorted, key=lambda x: x["metrics"]["directional_accuracy"])
console.print(f"\n2. BEST FOR TRADING (Directional Accuracy): [bold green]{best_dir['model_name'].upper()}[/bold green]")
console.print(f"   - Directional Accuracy: {best_dir['metrics']['directional_accuracy']:.2f}%")

# Best Sharpe
best_sharpe = max(evaluation_results_sorted, key=lambda x: x["metrics"]["fin_sharpe_ratio"])
console.print(f"\n3. BEST RISK-ADJUSTED (Sharpe Ratio): [bold green]{best_sharpe['model_name'].upper()}[/bold green]")
console.print(f"   - Sharpe Ratio: {best_sharpe['metrics']['fin_sharpe_ratio']:.2f}")

# Fastest training
fastest = min(training_results, key=lambda x: x["training_time"] if x["status"] == "success" else float('inf'))
console.print(f"\n4. FASTEST TRAINING: [bold green]{fastest['model_name'].upper()}[/bold green]")
console.print(f"   - Training time: {fastest['training_time']:.1f}s")

# Top 3 models
console.print(f"\n5. TOP 3 RECOMMENDED MODELS:")
for i, result in enumerate(evaluation_results_sorted[:3], 1):
    console.print(f"   {i}. {result['model_name']} (RMSE: {result['metrics']['rmse']:.2f}, Sharpe: {result['metrics']['fin_sharpe_ratio']:.2f})")

# Reality check
console.print(f"\n6. REALITY CHECK:")
avg_dir_acc = np.mean([r["metrics"]["directional_accuracy"] for r in evaluation_results_sorted])
if avg_dir_acc > SUSPICIOUS_DIR_ACC:
    console.print(f"   [red]✗ Average accuracy ({avg_dir_acc:.1f}%) is TOO HIGH - check for bugs[/red]")
elif avg_dir_acc > REALISTIC_DIR_ACC_MAX:
    console.print(f"   [yellow]⚠ Average accuracy ({avg_dir_acc:.1f}%) is above typical range[/yellow]")
else:
    console.print(f"   [green]✓ Average accuracy ({avg_dir_acc:.1f}%) is in realistic range (51-55%)[/green]")

# Next steps
console.print(f"\n7. NEXT STEPS:")
console.print(f"   → Use [bold]{best['model_name']}[/bold] for production predictions")
console.print(f"   → Run walk-forward validation: `price-stradamus train --model {best['model_name']}`")
console.print(f"   → Tune hyperparameters of top 3 models")
console.print(f"   → Increase n_epochs from {TRAINING_CONFIG['n_epochs']} to 100-200 for production")
console.print(f"   → See backtest_visualization.ipynb for extended analysis")

console.print("="*70)

======================================================================

MODEL COMPARISON - FINAL RECOMMENDATIONS

======================================================================

METHODOLOGY:

✓ Split RAW data first (no feature leakage)

...

→ Run walk-forward validation: `price-stradamus train --model xgboost`

→ Tune hyperparameters of top 3 models

→ Increase n_epochs from 200 to 100-200 for production

→ See backtest_visualization.ipynb for extended analysis

======================================================================